# 05 — Evaluation, error analysis, and Grad-CAM

Detailed evaluation of the trained arms on the **test split**, which has been
untouched until now — every selection decision in Notebooks 03 and 04 was made
on validation.

Three parts:

1. Confusion matrices, read hierarchically.
2. Per-class behaviour, with the minority classes called out.
3. Grad-CAM — evidence that the model attends to nucleus, chromatin, and
   cytoplasm rather than to background or staining artefacts.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src import cache, config, engine, gradcam, metrics, splits, transforms as T, viz
from src.dataset import MLL23Dataset
from src.models import HierarchicalClassifier
from src.hierarchy import BY_IDX, CLASSES, FINE_NAMES, LINEAGES

viz.apply_style()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

split_df = splits.load()
summary = pd.read_csv(engine.RESULTS_DIR / "summary.csv")
runs = summary[~summary.arm.str.startswith(("screen_", "smoke_"))].copy()
runs[["run_id", "arm", "seed", "test_macro_f1"]].head()

## Loading a trained arm

Checkpoints hold the weights selected on **validation macro F1**, not the final
epoch's weights.

In [ ]:
def load_arm(arm: str, seed: int = 0):
    """Restore the selected checkpoint for one arm, plus its config."""
    row = runs[(runs.arm == arm) & (runs.seed == seed)].iloc[0]
    ckpt = torch.load(engine.CHECKPOINT_DIR / row["run_id"] / "best.pt",
                      weights_only=True)
    model = HierarchicalClassifier(row["backbone"], mode=row["mode"],
                                   pretrained=False).to(DEVICE)
    model.load_state_dict(ckpt["model"])
    return model.eval(), row


def test_loader(stain_norm: bool = False, batch_size: int = 128):
    """Test-split loader reading the cache the arm was trained against."""
    want = cache.REINHARD_CACHE_PATH if stain_norm else cache.RAW_CACHE_PATH
    cp = cache.validate_cache(want, split_df)
    ds = MLL23Dataset(split_df[split_df.split == "test"], train=False, cache_path=cp)
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=engine.NUM_WORKERS, pin_memory=True)


ARMS_TO_EVAL = ["flat_baseline", "hierarchical", "no_imbalance", "stain_norm"]
preds = {}
for arm in ARMS_TO_EVAL:
    model, row = load_arm(arm)
    dl = test_loader(stain_norm=bool(row["stain_norm"]))
    # tta must match what the run recorded, or the numbers here will not
    # reconcile with results/summary.csv.
    preds[arm] = engine.predict(model, dl, DEVICE, tta=bool(row.get("tta", False)))
    del model; torch.cuda.empty_cache()
    print(f"  {arm:16s} done  (tta={bool(row.get('tta', False))})")

In [ ]:
# Reconciliation check: recomputed metrics must match the recorded run.
# A mismatch means the notebook and the results table describe different models.
for arm in ARMS_TO_EVAL:
    y2t, y2p, _, _ = preds[arm]
    here = metrics.classification_metrics(y2t, y2p)["macro_f1"]
    there = float(runs[(runs.arm == arm) & (runs.seed == 0)].iloc[0]["test_macro_f1"])
    flag = "OK" if abs(here - there) < 1e-6 else "MISMATCH"
    print(f"  {arm:16s} notebook {here:.4f}  summary.csv {there:.4f}  [{flag}]")

## 1. Confusion matrices

Class order is the **maturation continuum**, never alphabetical or
frequency-sorted. Adjacent maturation stages therefore sit adjacent to the
diagonal, so the clinically expected confusions appear as near-diagonal mass.
White rules mark the lineage boundaries: anything outside those blocks is a
severe cross-lineage error.

In [ ]:
for arm in ("flat_baseline", "hierarchical"):
    y2t, y2p, _, _ = preds[arm]
    cm = metrics.confusion(y2t, y2p, normalise=True)
    f = viz.confusion_heatmap(cm, title=f"Confusion matrix — {arm} (row-normalised)")
    f.savefig(config.ARTIFACT_DIR / f"confusion_{arm}.png")
plt.show()

In [ ]:
# Lineage-level view: the 3x3 summary of where cross-lineage error lands.
for arm in ("flat_baseline", "hierarchical"):
    y2t, y2p, _, _ = preds[arm]
    print(f"--- {arm} ---")
    print(metrics.lineage_confusion(y2t, y2p).round(3).to_string())
    print()

## 2. Hierarchical error decomposition

The direct test of the thesis. `within_error_share` is the number to watch: among
errors only, the fraction that stayed inside the correct lineage.

In [ ]:
rows = []
for arm, (y2t, y2p, y1t, y1p) in preds.items():
    r = metrics.hierarchical_errors(y2t, y2p)
    r["arm"] = arm
    rows.append(r)

pd.DataFrame(rows).set_index("arm").round(4)

## 3. Per-class behaviour

Macro averages hide exactly what matters here. This is the per-class view, with
minority classes marked.

In [ ]:
tables = {}
for arm, (y2t, y2p, _, _) in preds.items():
    tables[arm] = metrics.per_class_f1(y2t, y2p)

f = viz.per_class_f1_comparison(
    {k: tables[k] for k in ("flat_baseline", "hierarchical")},
    title="Per-class F1: flat baseline vs hierarchical  (* = minority class)")
f.savefig(config.ARTIFACT_DIR / "per_class_f1.png")
plt.show()

In [ ]:
# Side-by-side per-class F1, sorted by support so the tail is visible.
comp = tables["flat_baseline"][["class_name", "support", "is_minority"]].copy()
for arm in ARMS_TO_EVAL:
    comp[arm] = tables[arm]["f1"].round(3)
comp["hier - flat"] = (comp["hierarchical"] - comp["flat_baseline"]).round(3)
comp.sort_values("support")

**Reactive lymphocytes (33 images total, ~5 in test) deserve explicit
caution.** With a test support that small, per-class F1 moves in large
increments — a single prediction changing flips it substantially. Report the
number, but do not build an argument on its movement between arms.

## 4. Grad-CAM

CLAUDE.md treats these as a deliverable, not decoration: they are the evidence
that the model keys on cell morphology rather than on an acquisition confound. A
model with strong macro F1 and saliency sitting on the background has learned the
wrong thing, and only this figure reveals it.

Note on the ViT: its saliency is taken at `blocks[-1].norm1`, not at the final
block. timm's ViT pools with `global_pool='token'`, so at the last block the
patch tokens no longer influence the output and their gradient is exactly zero —
which would yield a flat map that still *looks* like a plausible figure.

In [ ]:
model, row = load_arm("hierarchical")
cache_path = cache.validate_cache(cache.RAW_CACHE_PATH, split_df)
test_df = split_df[split_df.split == "test"]

# One correctly-classified example per lineage, plus the rare classes.
show_idx = []
for lin_i, lin in enumerate(LINEAGES):
    sub = test_df[test_df.y1 == lin_i]
    show_idx.extend(sub.sample(2, random_state=0).index.tolist())
for cls_idx in sorted(metrics.MINORITY_IDX)[:6]:
    sub = test_df[test_df.y2 == cls_idx]
    if len(sub):
        # .index[0] keeps the manifest row label, which is what indexes the cache.
        show_idx.append(sub.sample(1, random_state=0).index[0])

panels_ds = MLL23Dataset(test_df.loc[show_idx], train=False, cache_path=cache_path)
batch = torch.stack([panels_ds[i][0] for i in range(len(panels_ds))]).to(DEVICE)
true_y2 = [int(panels_ds[i][2]) for i in range(len(panels_ds))]

In [ ]:
with gradcam.GradCAM(model, head="fine") as cam:
    heat = cam(batch)                       # saliency for the predicted class

_, pred_y2 = model.predict(batch)
pred_y2 = pred_y2.cpu().numpy()

panels = []
for i in range(len(batch)):
    ok = "OK" if pred_y2[i] == true_y2[i] else "MISS"
    caption = (f"{BY_IDX[true_y2[i]].name}\n-> {BY_IDX[int(pred_y2[i])].name} [{ok}]")
    panels.append((gradcam.overlay(batch[i], heat[i]), caption))

f = viz.gradcam_grid(panels, ncols=4,
                     title="Grad-CAM on the fine head (true -> predicted)")
f.savefig(config.ARTIFACT_DIR / "gradcam_fine.png")
plt.show()

### Fine head vs lineage head

Comparing where the two heads look shows whether the coarse and fine decisions
rest on the same evidence. Divergent saliency would suggest the heads have
learned separate features, which would undercut the "coarse decision regularises
the fine one" argument.

In [ ]:
with gradcam.GradCAM(model, head="lineage") as cam:
    heat_lin = cam(batch[:8])

panels = []
for i in range(8):
    panels.append((gradcam.overlay(batch[i], heat[i]),
                   f"fine: {BY_IDX[true_y2[i]].name}"))
    panels.append((gradcam.overlay(batch[i], heat_lin[i]),
                   f"lineage: {BY_IDX[true_y2[i]].lineage}"))

f = viz.gradcam_grid(panels, ncols=4, title="Fine head vs lineage head saliency")
f.savefig(config.ARTIFACT_DIR / "gradcam_heads.png")
plt.show()

## 5. Worst confusions

The specific class pairs the model conflates most, checked against the
biologically expected error mode. Adjacent maturation stages confusing with each
other is the *expected* pattern; cross-lineage pairs in this list are the ones to
explain.

In [ ]:
y2t, y2p, _, _ = preds["hierarchical"]
cm_counts = metrics.confusion(y2t, y2p, normalise=False).to_numpy()
np.fill_diagonal(cm_counts, 0)

pairs = []
for i in range(18):
    for j in range(18):
        if cm_counts[i, j] > 0:
            pairs.append({
                "true": FINE_NAMES[i],
                "predicted": FINE_NAMES[j],
                "count": int(cm_counts[i, j]),
                "share of true class": round(cm_counts[i, j] / max((y2t == i).sum(), 1), 3),
                "same lineage": BY_IDX[i].lineage == BY_IDX[j].lineage,
                "index gap": abs(i - j),
            })

pd.DataFrame(pairs).sort_values("count", ascending=False).head(15).reset_index(drop=True)

---

## Summary

All figures are written to `artifacts/`. The claims this notebook supports:

- Where each arm's errors fall, hierarchically — the within- vs cross-lineage
  split, which is the dissertation's central measurement.
- How the arms differ on the rare classes specifically, with the caveat that the
  rarest class has a test support of ~5 and should not carry an argument.
- That the model attends to cell morphology rather than background, via Grad-CAM
  on both heads.